In [13]:
############### import libraries ###############

import os
import json
import itertools

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.callbacks import EarlyStopping, Callback, ModelCheckpoint


In [14]:
############### path ###############

def download(url, output) :
    if not os.path.exists(output):
        print(f"{output} not found. Downloading...")
        urllib.request.urlretrieve(url, output)
    else:
        print(f"{output} already exists. Skip download.")

download("https://drive.google.com/uc?id=1JPOVfYXJNXeBgG_V0auiuK5W2CvP4_td", "training.csv")
download("https://drive.google.com/uc?id=11W8QiL98fEqV7Xfbw-xgXCC7G-SM1yMb", "test.csv")

training_path = 'training.csv'
test_path = 'test.csv'

SAVE_MODEL_DIR = "best_model"
model_name = "model.keras"

training.csv already exists. Skip download.
test.csv already exists. Skip download.


In [15]:
############### callback for AMS evaluation ###############

class AMSCallback(Callback):
    def __init__(self, X_val, y_val, w_val, w_Factor, n_thr = 201, pred_bs = 8192):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.w_val = w_val
        self.w_Factor = w_Factor
        self.n_thr = n_thr
        self.pred_bs = pred_bs

    def ams_val(self, s, b, b_r):
        radicand = 2.0 * ((s + b + b_r) * np.log(1.0 + s / (b + b_r)) - s)
        return np.sqrt(max(radicand, 0.0))

    def search_threshold(self, pred):
        best_ams = 0.0
        best_thr = 0.0
        for thr in np.linspace(0.0, 1.0, self.n_thr):
            s = self.w_val[(self.y_val == 1) & (pred >= thr)].sum() * self.w_Factor
            b = self.w_val[(self.y_val == 0) & (pred >= thr)].sum() * self.w_Factor
            score = self.ams_val(s, b, 10.0)
            if score > best_ams:
                best_ams = score
                best_thr = float(thr)
        return float(best_ams), float(best_thr)

    def on_epoch_end(self, epoch, logs = None):
        logs = logs or {}
        pred = self.model.predict(self.X_val, batch_size = self.pred_bs, verbose = 0).ravel()
        ams, thr = self.search_threshold(pred)
        logs["val_ams"] = ams
        logs["val_thr"] = thr
        print(f"Epoch {epoch + 1}:")
        print(f"val_AMS: {ams:.6f}, best_threshold: {thr:.6f}, val_loss: {logs['val_loss']:.3f}, val_auc: {logs['val_auc']:.3f}\n")

In [16]:
############### constants ###############

dp = 0.2   # dropout
lr = 5e-4  # learning_rate
wd = 5e-5  # weight_decay
bs = 1024  # model.fitのbatch_size
n_thr = 501 # thresholdの評価数 初めは粗く設定
n_thr_eval = 1001 # 最後はより細かく評価
base_units = 256

In [17]:
############### load training data ###############

df = pd.read_csv(training_path)

y = (df["Label"] == "s").astype(int).values

w = df["Weight"].values

X = df.drop(columns=["EventId", "Weight", "Label"]).values

X = np.where(X == -999.0, np.nan, X)
imputer = SimpleImputer(strategy = "median")
X = imputer.fit_transform(X)

In [18]:
############### training and evaluation ###############

X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X, y, w, test_size = 0.2, random_state = 1, stratify = y
)

scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_val = scalar.transform(X_val)

input_dim = X_train.shape[1]
wFactor = w.sum() /w_val.sum()

model = keras.Sequential([
    layers.Input(shape = (input_dim,)),
    layers.Dense(base_units, activation = "gelu"),
    layers.Dropout(dp),
    layers.Dense(base_units // 2, activation = "gelu"),
    layers.Dropout(dp),
    layers.Dense(base_units // 4, activation = "gelu"),
    layers.Dropout(dp),
    layers.Dense(1, activation = "sigmoid")
])

opt = keras.optimizers.AdamW(
    learning_rate = lr,
    weight_decay = wd
)

model.compile(
    optimizer = opt,
    loss = "binary_crossentropy",
    metrics = [keras.metrics.AUC(name="auc")]
)

ams_cb = AMSCallback(X_val, y_val, w_val, wFactor, n_thr = n_thr, pred_bs = 8192)

earlystopping = EarlyStopping(
    monitor = "val_ams",
    mode = "max",
    patience = 10,
    restore_best_weights = True
)

modelcheckpoint = ModelCheckpoint(
    model_name,
    monitor = "val_ams",
    mode = "max",
    save_best_only = True,
    verbose = 0
)

history = model.fit(
    X_train, y_train, sample_weight = w_train,
    validation_data = (X_val, y_val, w_val),
    epochs = 1000,
    batch_size = bs,
    verbose = 0,
    callbacks = [ams_cb, earlystopping, modelcheckpoint]
)

pred_val = model.predict(X_val, batch_size = 8192, verbose = 0).ravel()
ams_cb.n_thr = n_thr_eval
best_ams, best_thr = ams_cb.search_threshold(pred_val)
loss_val, auc_val = model.evaluate(X_val, y_val, sample_weight = w_val, verbose =0)

print(f"Best AMS: {best_ams:.7f} at threshold: {best_thr:.7f}, val_loss: {loss_val:.3f}, val_auc: {auc_val:.3f}")

Epoch 1:
val_AMS: 1.682704, best_threshold: 0.002000, val_loss: 0.019, val_auc: 0.654

Epoch 2:
val_AMS: 2.099855, best_threshold: 0.006000, val_loss: 0.018, val_auc: 0.711

Epoch 3:
val_AMS: 2.364745, best_threshold: 0.010000, val_loss: 0.017, val_auc: 0.740

Epoch 4:
val_AMS: 2.552049, best_threshold: 0.012000, val_loss: 0.016, val_auc: 0.761

Epoch 5:
val_AMS: 2.751591, best_threshold: 0.014000, val_loss: 0.016, val_auc: 0.779

Epoch 6:
val_AMS: 2.885759, best_threshold: 0.024000, val_loss: 0.016, val_auc: 0.790

Epoch 7:
val_AMS: 3.021905, best_threshold: 0.026000, val_loss: 0.016, val_auc: 0.799

Epoch 8:
val_AMS: 3.099599, best_threshold: 0.024000, val_loss: 0.016, val_auc: 0.800

Epoch 9:
val_AMS: 3.243804, best_threshold: 0.026000, val_loss: 0.016, val_auc: 0.806

Epoch 10:
val_AMS: 3.275035, best_threshold: 0.026000, val_loss: 0.016, val_auc: 0.817

Epoch 11:
val_AMS: 3.399323, best_threshold: 0.028000, val_loss: 0.015, val_auc: 0.818

Epoch 12:
val_AMS: 3.429125, best_thresho

In [19]:


# 1) ベストモデルをロード（ModelCheckpointで保存したやつ）
model = tf.keras.models.load_model(f"{model_name}")

# 2) ベース予測
pred_base = model.predict(X_val, batch_size=8192, verbose=0).ravel()

# 3) ベースAMS（粗めでOK）
ams_cb.n_thr = 201
base_ams, base_thr = ams_cb.search_threshold(pred_base)
print("base AMS:", base_ams, "thr:", base_thr)

# 4) permutation importance（AMS低下量）
D = X_val.shape[1]
drops = np.zeros(D, dtype=np.float64)

rng = np.random.default_rng(42)

for j in range(D):
    Xp = X_val.copy()
    perm = rng.permutation(Xp.shape[0])
    Xp[:, j] = Xp[perm, j]

    pred_p = model.predict(Xp, batch_size=8192, verbose=0).ravel()
    ams_p, _ = ams_cb.search_threshold(pred_p)

    drops[j] = base_ams - ams_p
    # print(j, drops[j])  # 必要なら進捗表示

# 重要度ランキング（落ち幅が大きい順）
top = np.argsort(-drops)[:20]
print("Top features by AMS drop:")
for k in top:
    print(k, drops[k])


base AMS: 3.6988807441609857 thr: 0.03
Top features by AMS drop:
2 1.829697211710851
13 1.5007418887888466
7 1.4554602201847184
0 1.2765879111274527
1 1.2304711650597855
5 0.8422625738815932
10 0.7386212707470805
23 0.6780132756066743
19 0.6071342350759346
16 0.40450043971549254
22 0.3529980488434279
3 0.34276162376027175
21 0.2887084231443202
11 0.2726640022985314
24 0.2368130199552949
8 0.22094261815637228
17 0.1849132779142928
29 0.17039222614715843
14 0.10363292359763099
4 0.08945496295998367
